In [2]:
!pip install transformers -q
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [3]:
model_name = "distilbert-base-uncased-finetuned-sst-2-english"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [4]:
model.eval()

text = "I love this movie!"
inputs = tokenizer(text, return_tensors="pt")

print(inputs)

{'input_ids': tensor([[ 101, 1045, 2293, 2023, 3185,  999,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1]])}


In [5]:
model.eval() #switching to eval mode now

with torch.no_grad():
    outputs = model(**inputs)

print(outputs)

SequenceClassifierOutput(loss=None, logits=tensor([[-4.3246,  4.6837]]), hidden_states=None, attentions=None)


In [6]:
logits = outputs.logits #these logits give us the final numbers after calculations
print(logits)

tensor([[-4.3246,  4.6837]])


In [7]:
# Softmax conversion to get percentages

probabilities = torch.softmax(logits, dim=-1) #softmax converts the logits into percentages
print(probabilities)

tensor([[1.2238e-04, 9.9988e-01]])


In [8]:
predicted_class_id = torch.argmax(probabilities, dim=-1).item()
print(predicted_class_id)

print(model.config.id2label) #assigning negative and positive

predicted_label = model.config.id2label[predicted_class_id]
print(predicted_label)

1
{0: 'NEGATIVE', 1: 'POSITIVE'}
POSITIVE


In [9]:
confidence = probabilities[0][predicted_class_id].item()
print(f"Label: {predicted_label}, Score: {confidence:.4f}")

Label: POSITIVE, Score: 0.9999


In [10]:
from transformers import pipeline
clf = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")
print(clf("I love this movie!"))

# manual path, same sentence, for comparison
print(f"manual: {predicted_label}, score: {confidence:.4f}")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'POSITIVE', 'score': 0.9998775720596313}]
manual: POSITIVE, score: 0.9999
